# Unsupervised Learning: k-Means, Hierarchical Clustering and PCA – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers or simulated data – no data set is loaded.

**Contents**
1. Unsupervised learning: no target variable
2. k-means clustering
3. Choosing k: elbow method and silhouette analysis
4. Hierarchical clustering and the dendrogram
5. Principal component analysis (PCA)

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. Unsupervised learning: no target variable

| | supervised (Parts 3–4) | unsupervised (Part 5) |
|---|---|---|
| data | features $x$ **and** known answer $y$ | only features $x$ |
| goal | predict $y$ for new cases | find groups or a compact description |
| evaluation | test error (RMSE, accuracy, AUC) | no target used for fitting: internal criteria, stability, interpretation (external labels sometimes available) |
| typical use | rent prediction, churn yes/no | customer segments, anomaly detection, visualisation |

- **Clustering:** group similar observations – k-means, hierarchical clustering.
- **Dimension reduction:** summarise many features by a few new ones – PCA.

## 2. k-means clustering

1. Choose the number of clusters $k$.
2. Place $k$ centres at random.
3. Assign every point to its nearest centre.
4. Move each centre to the mean of its points – repeat 3–4 until (almost) nothing changes.

In words: k-means makes the distances of the points to their own cluster centre as small as possible, i.e. it minimises
the **within-cluster sum of squares**

$$\text{WCSS} = \sum_{c=1}^{k}\sum_{x_i \in C_c}\lVert x_i - \mu_c\rVert^2 .$$

In Python: `KMeans(n_clusters=3, n_init=10)`. The result depends on the random start (local optimum) → several starts
(`n_init`), the best solution (smallest WCSS) is kept.

In [ ]:
# k-means on three simulated clusters
X, _ = make_blobs(
    n_samples=150,
    centers=[[-2, -1.5], [2.5, -1], [0, 2.5]],
    cluster_std=0.8,
    random_state=4,
)
centres = np.array(
    [[-1.5, -2.5], [-0.5, -0.5], [2.5, 2.5]]
)  # deliberately bad start

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, it, title in [
    (axes[0], 0, 'Start: random centres'),
    (axes[1], 1, 'After 1 iteration'),
    (axes[2], 20, 'Converged'),
]:
    c = centres.copy()
    for _ in range(it):
        labels = np.argmin(
            ((X[:, None, :] - c[None, :, :]) ** 2).sum(axis=2), axis=1
        )
        c = np.array([X[labels == j].mean(axis=0) for j in range(3)])
    labels = np.argmin(
        ((X[:, None, :] - c[None, :, :]) ** 2).sum(axis=2), axis=1
    )
    colors = np.array(['tab:blue', 'tab:orange', 'tab:green'])
    ax.scatter(
        X[:, 0], X[:, 1], s=12, color='grey' if it == 0 else colors[labels]
    )
    ax.scatter(
        c[:, 0], c[:, 1], s=250, marker='X', c=colors, edgecolors='black'
    )
    wcss = ((X - c[labels]) ** 2).sum()
    ax.set_title(f'{title} (WCSS = {wcss:.0f})')
plt.show()

## 3. Choosing k: elbow method and silhouette analysis

**Reading the elbow plot**
- y-axis: how spread out the points are within their clusters (WCSS; smaller = tighter clusters),
- more clusters always reduce it – $k = n$ would give 0,
- choose $k$ where the curve bends (the "elbow").

**Caution:** scale the features first, use several random starts, and check that the clusters make sense for the
business question.

**Additional explanation – silhouette score.** For each point, $s = \frac{b - a}{\max(a, b)}$ with $a$ = mean distance to
the points of its own cluster and $b$ = mean distance to the points of the nearest other cluster. $s$ close to 1: well
clustered; close to 0: between two clusters; negative: probably in the wrong cluster. The mean silhouette over all points
can be compared across different $k$ (higher is better).

In [ ]:
# Elbow method and silhouette score for k = 1..8
ks = range(1, 9)
wcss = [
    KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).inertia_ for k in ks
]
sil = [
    silhouette_score(
        X, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    )
    for k in ks[1:]
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, wcss, marker='o')
axes[0].scatter(
    [3], [wcss[2]], s=200, color='tab:orange', zorder=3, label='elbow: k = 3'
)
axes[0].set_xlabel('number of clusters k')
axes[0].set_ylabel('within-cluster sum of squares')
axes[0].set_title('Elbow plot')
axes[0].legend()
axes[1].plot(list(ks)[1:], sil, marker='o', color='darkred')
axes[1].set_xlabel('number of clusters k')
axes[1].set_ylabel('mean silhouette score')
axes[1].set_title('Silhouette score')
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

**Silhouette analysis.** The silhouette plot shows the silhouette value of **every** point, sorted within each cluster.
Wide bars above the mean line indicate well-separated clusters; thin clusters, many values close to 0 or negative values
indicate a poor choice of $k$ or badly placed points. Below: $k = 3$ (the true structure) vs. $k = 5$.

In [ ]:
# Silhouette plots for k = 3 and k = 5
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
for ax, k in zip(axes, [3, 5]):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    s = silhouette_samples(X, labels)
    y = 0
    for c in range(k):
        vals = np.sort(s[labels == c])
        ax.fill_betweenx(
            np.arange(y, y + len(vals)),
            0,
            vals,
            color=plt.colormaps['tab10'](c),
            alpha=0.8,
        )
        ax.text(-0.05, y + len(vals) / 2, str(c))
        y += len(vals) + 5
    ax.axvline(
        s.mean(),
        color='black',
        linestyle='--',
        label=f'mean silhouette = {s.mean():.2f}',
    )
    ax.set_title(f'Silhouette plot, k = {k}')
    ax.set_xlabel('silhouette value')
    ax.set_yticks([])
    ax.legend(loc='lower right')
plt.show()

## 4. Hierarchical clustering and the dendrogram

**Agglomerative clustering:**
1. Start: every observation (e.g. every customer) is its own cluster.
2. Merge the two closest clusters; repeat until one cluster is left.
3. Cut the **dendrogram** at a height → clusters.

The height of a merge in the dendrogram is the distance between the merged clusters. Long vertical lines mean that
clearly separated groups are merged only late – a good place to cut.

**Additional explanation – linkage.** How is the distance between two *clusters* defined? *Single linkage*: closest pair
of points; *complete linkage*: farthest pair; *average linkage*: mean of all pairwise distances; *Ward*: merge the two
clusters that increase the WCSS the least (similar to k-means, often the default choice).

In [ ]:
# Hierarchical clustering (Ward) of twelve customers
rng = np.random.default_rng(5)
customers = np.vstack(
    [
        rng.normal([1, 1], 0.4, (4, 2)),
        rng.normal([4, 4], 0.3, (4, 2)),
        rng.normal([6, 1.5], 0.4, (4, 2)),
    ]
)
names = list('ABCDEFGHIJKL')

Z = linkage(customers, method='ward')
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].scatter(customers[:, 0], customers[:, 1], s=60, color='black')
for (x0, y0), name in zip(customers, names):
    axes[0].annotate(name, (x0, y0), xytext=(5, 5), textcoords='offset points')
axes[0].set_title('12 customers (2 features)')
dendrogram(Z, labels=names, ax=axes[1], color_threshold=3)
axes[1].axhline(
    3, color='tab:orange', linestyle='--', label='cut here → 3 clusters'
)
axes[1].set_ylabel('merge distance')
axes[1].set_title('Dendrogram (Ward linkage)')
axes[1].legend()
plt.show()
print(
    'Clusters after the cut:',
    dict(zip(names, fcluster(Z, t=3, criterion='distance').tolist())),
)

## 5. Principal component analysis (PCA)

**Idea:** PCA rotates the coordinate system so that the first new axis (**PC1**) captures as much of the spread
(variance) as possible; PC2 captures as much as possible of the remaining spread, orthogonal to PC1; and so on.

- The **loadings** (`pca.components_`) show how each principal component is composed of the original features.
- The **explained variance ratio** shows which share of the total variance each component captures.
- Features measured on different scales must be **standardised** first – otherwise PCA is dominated by the features with
  the largest variance.

Example: two strongly correlated features – PC1 explains about 86 % of the total variance, so the two features can be summarised well by one.

In [ ]:
# PCA of two correlated features
rng = np.random.default_rng(0)
f1 = rng.normal(0, 1.8, 200)
f2 = 0.65 * f1 + rng.normal(0, 1.0, 200)
F = np.c_[f1, f2]

pca = PCA().fit(F)
evr = pca.explained_variance_ratio_
print('explained variance ratio:', evr.round(3))
print(
    'loadings PC1:',
    pca.components_[0].round(2),
    ' PC2:',
    pca.components_[1].round(2),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(F[:, 0], F[:, 1], s=12, alpha=0.6)
for comp, var, name, color in zip(
    pca.components_,
    pca.explained_variance_,
    ['PC1', 'PC2'],
    ['tab:orange', 'navy'],
):
    v = comp * 2 * np.sqrt(var)
    axes[0].annotate(
        '',
        xy=pca.mean_ + v,
        xytext=pca.mean_,
        arrowprops={'arrowstyle': '->', 'color': color, 'lw': 2.5},
    )
    axes[0].text(*(pca.mean_ + v * 1.15), name, color=color, fontweight='bold')
axes[0].set_xlabel('feature 1')
axes[0].set_ylabel('feature 2')
axes[0].set_title('New axes along the largest spread')
axes[0].axis('equal')
axes[1].bar(['PC1', 'PC2'], evr * 100, color=['tab:orange', 'navy'])
for i, v in enumerate(evr * 100):
    axes[1].text(i, v + 1, f'{v:.0f} %', ha='center')
axes[1].set_ylabel('share of total variance [%]')
axes[1].set_title('Explained variance')
plt.show()

**Key messages**
- Unsupervised learning has no target: results are judged by internal criteria, stability and interpretation.
- k-means minimises the WCSS; scale the features, use several starts and choose $k$ with the elbow plot, the silhouette
  score and domain knowledge.
- Hierarchical clustering shows the whole merge history in a dendrogram; cutting it gives the clusters.
- PCA summarises many correlated features by a few principal components.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')